<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/%EC%96%91%EA%B7%BC%EC%98%81%ED%95%99%EC%83%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# data.yaml 예시
path: ./datasets
train: images/train
val: images/val

nc: 2
names: ['lane', 'traffic_sign']

datasets/
├── images/
│   ├── train/
│   └── val/
├── labels/
│   ├── train/
│   └── val/

model.train(
    data="/content/dataset/dataset.yaml",  # yaml 경로
    epochs=120,        # 학습 에폭 수
    imgsz=640,         # 입력 이미지 크기
    batch=16,          # 배치 사이즈
    name="lane_model",  # 저장 폴더 이름
)

from ultralytics import YOLO

# 모델 로드
model = YOLO('runs/detect/train/weights/best.pt')

# 이미지에 대한 예측
results = model.predict(source='sample.jpg')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ZIP 파일을 코랩으로 복사
!cp "/content/drive/MyDrive/6_23_Lesson/dataset.zip" "/content/"

# 압축 해제
!unzip -o /content/dataset.zip -d /content/

# 압축 해제 확인
!ls -la /content/

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [ ]:
import torch.nn as nn

class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),  # [B, 3, 224, 224] → [B, 16, 224, 224]
            nn.ReLU(),
            nn.MaxPool2d(2),                             # → [B, 16, 112, 112]

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                             # → [B, 32, 56, 56]

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))                 # → [B, 64, 1, 1]
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x


In [ ]:
# ✅ 설정
data_dir = '/content/data'
batch_size = 16
num_epochs = 100
lr = 0.001
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ 데이터셋
train_dataset = datasets.ImageFolder(f"{data_dir}/train", transform=transform)
val_dataset = datasets.ImageFolder(f"{data_dir}/val", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

num_classes = len(train_dataset.classes)

# ✅ 모델 정의
model = CustomCNN(num_classes=num_classes).to(device)

# ✅ 손실 함수 & 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# ✅ 학습 루프
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[Epoch {epoch+1}/{num_epochs}] Loss: {total_loss:.4f}")

# ✅ 저장
torch.save(model.state_dict(), "custom_cnn.pth")
print("✅ 커스텀 모델 저장 완료")


톻합코드 짧은거

In [ ]:
import os
import shutil

# 기본 클래스 폴더 생성
os.makedirs('/content/dataset/train/images/default', exist_ok=True)
os.makedirs('/content/dataset/valid/images/default', exist_ok=True)

# train 이미지들을 default 폴더로 이동
train_images_path = '/content/dataset/train/images'
png_files = [f for f in os.listdir(train_images_path) if f.endswith('.png')]

for file in png_files:
    shutil.move(
        os.path.join(train_images_path, file),
        os.path.join(train_images_path, 'default', file)
    )

print(f"✅ {len(png_files)}개 파일을 default 폴더로 이동완료")

# valid 폴더도 같은 작업 (있다면)
valid_images_path = '/content/dataset/valid/images'
if os.path.exists(valid_images_path):
    valid_files = [f for f in os.listdir(valid_images_path) if f.endswith('.png')]
    for file in valid_files:
        shutil.move(
            os.path.join(valid_images_path, file),
            os.path.join(valid_images_path, 'default', file)
        )
    print(f"✅ {len(valid_files)}개 valid 파일도 이동완료")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import gc

# 메모리 정리 함수
def clear_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

# 간단한 CNN 모델
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 28 * 28, 512)
        self.fc2 = nn.Linear(512, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(-1, 128 * 28 * 28)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# ✅ 설정
data_dir = '/content/dataset'
batch_size = 16
num_epochs = 100
lr = 0.001
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ 데이터셋
#train_dataset = datasets.ImageFolder(f"{data_dir}/train", transform=transform)
#val_dataset = datasets.ImageFolder(f"{data_dir}/valid", transform=transform)


train_dataset = datasets.ImageFolder(f"{data_dir}/train/images", transform=transform)
val_dataset = datasets.ImageFolder(f"{data_dir}/valid/images", transform=transform)  # valid 폴더도 같은 구조일 것

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

num_classes = len(train_dataset.classes)

# ✅ 모델 정의
model = CustomCNN(num_classes=num_classes).to(device)

# ✅ 손실 함수 & 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# ✅ 학습 루프
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[Epoch {epoch+1}/{num_epochs}] Loss: {total_loss:.4f}")

    # 메모리 정리 (10 에포크마다)
    if epoch % 10 == 0:
        clear_gpu_memory()

# ✅ 저장
torch.save(model.state_dict(), "custom_cnn.pth")
print("✅ 커스텀 모델 저장 완료")

톻합코드

In [ ]:
# Google Colab용 학습 코드

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os
import gc
import matplotlib.pyplot as plt

# GPU 메모리 정리 함수
def clear_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()
    print("🧹 GPU Memory Cleared")

# CustomCNN 모델 정의
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)

        # Pooling layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Dropout
        self.dropout = nn.Dropout(0.5)

        # Fully connected layers
        self.fc1 = nn.Linear(128 * 28 * 28, 512)  # 224/8 = 28
        self.fc2 = nn.Linear(512, num_classes)

        # Activation
        self.relu = nn.ReLU()

    def forward(self, x):
        # Convolutional layers with pooling
        x = self.pool(self.relu(self.conv1(x)))  # 224 -> 112
        x = self.pool(self.relu(self.conv2(x)))  # 112 -> 56
        x = self.pool(self.relu(self.conv3(x)))  # 56 -> 28

        # Flatten
        x = x.view(-1, 128 * 28 * 28)

        # Fully connected layers
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

# ✅ 코랩 설정
print("🚀 Google Colab 환경 확인...")
print(f"Python version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# 데이터 경로 설정 (코랩용)
data_dir = '/content/dataset'  # 압축 해제된 dataset 폴더
batch_size = 16  # 코랩 메모리에 맞게 조정
num_epochs = 100
lr = 0.001
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🔥 Using device: {device}")

# 초기 메모리 정리
clear_gpu_memory()

# ✅ 데이터셋 경로 확인
print("\n📁 Dataset structure check:")
print(f"Dataset dir: {data_dir}")

# 폴더 구조 확인
if os.path.exists(data_dir):
    print("✅ Dataset folder found")
    subdirs = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    print(f"📊 Subdirectories: {subdirs}")

    # train 폴더 확인
    train_path = f"{data_dir}/train"
    if os.path.exists(train_path):
        print("✅ Train folder found")
        train_classes = [d for d in os.listdir(train_path) if os.path.isdir(os.path.join(train_path, d))]
        print(f"🏷️ Train classes: {train_classes}")
        print(f"📊 Number of classes: {len(train_classes)}")
    else:
        print("❌ Train folder not found!")

    # valid 폴더 확인 (valid 또는 val)
    valid_path = f"{data_dir}/valid"
    val_path = f"{data_dir}/val"

    if os.path.exists(valid_path):
        print("✅ Valid folder found")
        val_classes = [d for d in os.listdir(valid_path) if os.path.isdir(os.path.join(valid_path, d))]
        print(f"🏷️ Valid classes: {val_classes}")
        validation_folder = "valid"
    elif os.path.exists(val_path):
        print("✅ Val folder found")
        val_classes = [d for d in os.listdir(val_path) if os.path.isdir(os.path.join(val_path, d))]
        print(f"🏷️ Val classes: {val_classes}")
        validation_folder = "val"
    else:
        print("❌ Validation folder not found!")
        validation_folder = None
else:
    print("❌ Dataset folder not found!")
    print("🔍 Available folders in /content/:")
    print([d for d in os.listdir('/content/') if os.path.isdir(f'/content/{d}')])

# 데이터셋 로딩 중단점
if not os.path.exists(train_path) or validation_folder is None:
    print("💡 Please check your dataset structure!")
    print("Expected structure:")
    print("/content/dataset/")
    print("  ├── train/")
    print("  │   ├── class1/")
    print("  │   └── class2/")
    print("  └── valid/ (or val/)")
    print("      ├── class1/")
    print("      └── class2/")
    exit()

# ✅ 전처리 (코랩용 최적화)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ✅ 데이터셋 로드
try:
    train_dataset = datasets.ImageFolder(f"{data_dir}/train", transform=transform)
    val_dataset = datasets.ImageFolder(f"{data_dir}/{validation_folder}", transform=transform)

    print(f"\n✅ Dataset loaded successfully!")
    print(f"📊 Train dataset size: {len(train_dataset)}")
    print(f"📊 Valid dataset size: {len(val_dataset)}")
    print(f"🏷️ Classes: {train_dataset.classes}")

except Exception as e:
    print(f"❌ Dataset loading failed: {e}")
    exit()

# DataLoader 생성 (코랩 메모리 최적화)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

num_classes = len(train_dataset.classes)

# ✅ 모델 정의
model = CustomCNN(num_classes=num_classes).to(device)
print(f"\n🚀 Model created with {num_classes} classes")

# 모델 파라미터 수 확인
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"📊 Total parameters: {total_params:,}")
print(f"📊 Trainable parameters: {trainable_params:,}")

# ✅ 손실 함수 & 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# Learning rate scheduler 추가 (선택사항)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

# ✅ 학습 루프 (코랩 최적화)
print(f"\n🔥 Starting training for {num_epochs} epochs on {device}...")
print("=" * 60)

best_val_acc = 0.0
train_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Training accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

        # 메모리 정리 (코랩용)
        if batch_idx % 20 == 0:
            torch.cuda.empty_cache()

    # Calculate training metrics
    epoch_loss = running_loss / len(train_loader)
    train_acc = 100 * correct_train / total_train
    train_losses.append(epoch_loss)

    # Validation phase
    model.eval()
    correct_val = 0
    total_val = 0
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_acc = 100 * correct_val / total_val
    val_loss = val_loss / len(val_loader)
    val_accuracies.append(val_acc)

    # Learning rate scheduling
    scheduler.step()

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/content/best_custom_cnn.pth")
        print(f"💾 New best model saved! Val Acc: {val_acc:.2f}%")

    # Print progress (코랩용 깔끔한 출력)
    print(f"Epoch [{epoch+1:3d}/{num_epochs}] "
          f"Loss: {epoch_loss:.4f} | "
          f"Train Acc: {train_acc:.2f}% | "
          f"Val Acc: {val_acc:.2f}% | "
          f"LR: {optimizer.param_groups[0]['lr']:.6f}")

    # 메모리 정리 (20 에포크마다)
    if (epoch + 1) % 20 == 0:
        clear_gpu_memory()
        print(f"🧹 Memory cleaned at epoch {epoch + 1}")

# ✅ 최종 모델 저장
torch.save(model.state_dict(), "/content/final_custom_cnn.pth")

print("\n" + "=" * 60)
print(f"✅ Training completed!")
print(f"🏆 Best validation accuracy: {best_val_acc:.2f}%")
print(f"💾 Models saved in /content/:")
print(f"  - best_custom_cnn.pth (best validation accuracy)")
print(f"  - final_custom_cnn.pth (final epoch)")

# 최종 메모리 정리
clear_gpu_memory()

# ✅ 성능 그래프 출력 (코랩용)
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(train_losses, 'b-', label='Training Loss')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(val_accuracies, 'r-', label='Validation Accuracy')
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
epochs = range(1, len(train_losses) + 1)
plt.plot(epochs, train_losses, 'b-', label='Training Loss')
plt.plot(epochs, [acc/100 for acc in val_accuracies], 'r-', label='Validation Accuracy (scaled)')
plt.title('Training Progress')
plt.xlabel('Epoch')
plt.ylabel('Loss / Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# 최종 요약
print(f"\n📊 Training Summary:")
print(f"  🎯 Final Training Loss: {train_losses[-1]:.4f}")
print(f"  🎯 Final Validation Accuracy: {val_accuracies[-1]:.2f}%")
print(f"  🏆 Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"  ⏱️ Total Epochs: {num_epochs}")
print(f"  🔧 Batch Size: {batch_size}")
print(f"  📚 Number of Classes: {num_classes}")
print(f"  🏷️ Class Names: {train_dataset.classes}")

print("\n🎉 All done! Your model is ready to use!")

# 모델 로드 예시 코드 출력
print("\n💡 To load your trained model later:")
print("```python")
print("model = CustomCNN(num_classes=" + str(num_classes) + ")")
print("model.load_state_dict(torch.load('/content/best_custom_cnn.pth'))")
print("model.eval()")
print("```")